In [13]:
import os
import json
import numpy as np
import pandas as pd
import tensorflow as tf
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam


In [14]:
# Define dataset paths
json_path = r"D:\College\Term3\computervision\Yelp-Photos\Yelp Photos\yelp_photos\photos.json"
image_dir = r"D:\College\Term3\computervision\Yelp-Photos\Yelp Photos\yelp_photos\photos"

# Load JSON file
df = pd.read_json(json_path, lines=True)

# Select a subset of images (modify if needed)
NUM_IMAGES = 10000  # Load 10,000 images
df = df.sample(n=min(NUM_IMAGES, len(df)), random_state=42)

# Create full image paths
df['image_path'] = df['photo_id'].apply(lambda x: os.path.join(image_dir, f"{x}.jpg"))

# Remove missing files
df = df[df['image_path'].apply(os.path.exists)]

print(f"✅ Loaded {len(df)} valid images.")


✅ Loaded 10000 valid images.


In [ ]:
IMG_SIZE = (224, 224)  # Required size for VGG16

def preprocess_image(image_path):
    """Load, resize, and normalize image."""
    try:
        img = Image.open(image_path).convert('RGB')
        img = img.resize(IMG_SIZE)  # Resize to 224x224
        return np.array(img).astype(np.float32) / 255.0  # Normalize to [0,1]
    except:
        return None

# Apply preprocessing
df['image_array'] = df['image_path'].apply(preprocess_image)

# Drop rows where images couldn't be processed
df.dropna(subset=['image_array'], inplace=True)

print(f"✅ Image preprocessing complete! Total processed images: {len(df)}")


In [5]:
def augment_image(image_array):
    """Apply Gaussian Blur and Histogram Equalization."""
    try:
        img = (image_array * 255).astype(np.uint8)  # Convert back to uint8 for OpenCV

        # Apply Gaussian Blur
        img = cv2.GaussianBlur(img, (3, 3), 0)

        # Convert to YUV for Histogram Equalization
        img_yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
        img_yuv[:, :, 0] = cv2.equalizeHist(img_yuv[:, :, 0])
        img = cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)

        return img.astype(np.float32) / 255.0  # Convert back to float32 & normalize
    except:
        return None

# Apply augmentation
df['augmented_image'] = df['image_array'].apply(augment_image)

# Drop failed augmentations
df.dropna(subset=['augmented_image'], inplace=True)

print(f"✅ Image augmentation complete! Total augmented images: {len(df)}")


✅ Image augmentation complete! Total augmented images: 9998


In [6]:
import tensorflow as tf

# Function to load and preprocess images dynamically
def load_image(image_path, label):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)  # Decode JPG image
    image = tf.image.resize(image, [224, 224])  # Resize for VGG16
    image = image / 255.0  # Normalize to [0,1]
    return image, label

# Convert labels to TensorFlow tensors
label_encoder = LabelEncoder()
df['encoded_label'] = label_encoder.fit_transform(df['label'])

# Create TensorFlow dataset
image_paths = df['image_path'].values  # Image file paths
labels = df['encoded_label'].values  # Encoded labels

# Create TensorFlow dataset pipeline (Efficient, No RAM Overload)
dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.shuffle(buffer_size=len(df))  # Shuffle for randomness
dataset = dataset.batch(32).prefetch(tf.data.AUTOTUNE)  # Batch processing

# Split into training & testing datasets (80% train, 20% test)
train_size = int(0.8 * len(df))
train_dataset = dataset.take(train_size)  # First 80% for training
test_dataset = dataset.skip(train_size)  # Remaining 20% for testing

print(f"✅ TensorFlow dataset created! Training samples: {train_size}, Testing samples: {len(df) - train_size}")


✅ TensorFlow dataset created! Training samples: 7998, Testing samples: 2000


In [7]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

# Load VGG16 model (without top classification layers)
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze base model layers
base_model.trainable = False

# Print model summary
base_model.summary()


Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [13]:
# Add custom classification layers
x = Flatten()(base_model.output)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output_layer = Dense(len(label_encoder.classes_), activation="softmax")(x)  # Softmax for multi-class

# Define full model
model = Model(inputs=base_model.input, outputs=output_layer)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Print summary
model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [9]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout
from tensorflow.keras.optimizers import Adam

# Load VGG16 model (excluding top classification layers)
base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze base model layers
base_model.trainable = False

# Print model summary
base_model.summary()


Model: "vgg16"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [11]:
# Add custom classification layers
x = Flatten()(base_model.output)
x = Dense(512, activation="relu")(x)
x = Dropout(0.5)(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.3)(x)
output_layer = Dense(len(label_encoder.classes_), activation="softmax")(x)  # Softmax for multi-class

# Define full model
model = Model(inputs=base_model.input, outputs=output_layer)

# Compile the model
model.compile(optimizer=Adam(learning_rate=0.0001),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

# Print summary
model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 224, 224, 3)]     0         
                                                                 
 block1_conv1 (Conv2D)       (None, 224, 224, 64)      1792      
                                                                 
 block1_conv2 (Conv2D)       (None, 224, 224, 64)      36928     
                                                                 
 block1_pool (MaxPooling2D)  (None, 112, 112, 64)      0         
                                                                 
 block2_conv1 (Conv2D)       (None, 112, 112, 128)     73856     
                                                                 
 block2_conv2 (Conv2D)       (None, 112, 112, 128)     147584    
                                                                 
 block2_pool (MaxPooling2D)  (None, 56, 56, 128)       0     

In [12]:
history = model.fit(
    train_dataset.batch(8),  # Reduce batch size
    validation_data=test_dataset.batch(8),
    epochs=5,
    verbose=1
)

# Save model after initial training
model.save("vgg16_feature_extraction.h5")

print("✅ Initial Training Complete!")


Epoch 1/5


ValueError: in user code:

    File "c:\Users\tirth\anaconda3\Lib\site-packages\keras\engine\training.py", line 1284, in train_function  *
        return step_function(self, iterator)
    File "c:\Users\tirth\anaconda3\Lib\site-packages\keras\engine\training.py", line 1268, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\tirth\anaconda3\Lib\site-packages\keras\engine\training.py", line 1249, in run_step  **
        outputs = model.train_step(data)
    File "c:\Users\tirth\anaconda3\Lib\site-packages\keras\engine\training.py", line 1050, in train_step
        y_pred = self(x, training=True)
    File "c:\Users\tirth\anaconda3\Lib\site-packages\keras\utils\traceback_utils.py", line 70, in error_handler
        raise e.with_traceback(filtered_tb) from None
    File "c:\Users\tirth\anaconda3\Lib\site-packages\keras\engine\input_spec.py", line 298, in assert_input_compatibility
        raise ValueError(

    ValueError: Input 0 

In [10]:
history = model.fit(
    train_dataset.batch(8),  # Reduce batch size
    validation_data=test_dataset.batch(8),
    epochs=5,
    verbose=1
)

# Save model after initial training
model.save("vgg16_feature_extraction.h5")

print("✅ Initial Training Complete!")


NameError: name 'model' is not defined